# Multiple Linear Regression and Dummy Variables

**STA 506 2.0 Linear Regression Analysis** · [Subject index](00_Subject_Index.ipynb)

## Learning route

1. Read the theory and definitions in this notebook.
2. Explain the symbols and assumptions.
3. Predict, calculate, then run the simulation.
4. Change one parameter and explain the result.
5. Complete the practice before revealing answers.

![Visual route](../_images/sta506.svg)

## Theory contents

These notes are written for this course in our own words. Theory, worked tables, code and exercises are all in this notebook; no PDF pages are embedded.

- [Conditional effects](#theory-01)
- [Design matrix and estimation](#theory-02)
- [Variance and tests](#theory-03)
- [Indicator variables](#theory-04)
- [Interactions and separate equations](#theory-05)
- [Fit, prediction and model selection](#theory-06)
- [Reporting](#theory-07)

## Detailed theory · Multiple regression and indicator variables

<a id="theory-01"></a>

### Conditional effects
The model $Y=X\beta+\epsilon$ includes p coefficient columns, usually including an intercept. A slope measures the change in conditional mean per predictor unit while the other included predictors are held fixed. It is not automatically a marginal association or causal effect.

<a id="theory-02"></a>

### Design matrix and estimation
Rows correspond to observations and columns to model terms. With full column rank,
$$\hat\beta=(X^\top X)^{-1}X^\top y,\quad \hat y=X\hat\beta,\quad SSE=(y-\hat y)^\top(y-\hat y).$$
Use a stable least-squares solver in code. Collinear columns prevent unique estimation; near-collinearity inflates uncertainty and can make coefficients unstable. More observations do not remove exact column redundancy.

<a id="theory-03"></a>

### Variance and tests
$$s^2=SSE/(n-p),\quad \widehat{Cov}(\hat\beta)=s^2(X^\top X)^{-1}.$$
The diagonal supplies coefficient variances. Individual t tests use n−p df. For a full model and nested reduced model,
$$F=\frac{(SSE_R-SSE_F)/(p_F-p_R)}{SSE_F/(n-p_F)}.$$
This tests a collection of restrictions and need not agree with informal interpretation of separate coefficient p-values.

<a id="theory-04"></a>

### Indicator variables
For k categories, an intercept model usually needs k−1 indicators. The omitted category is the reference. With $\beta_0+\beta_1x+\gamma D$, groups have different intercepts but a common slope. Including every category indicator plus an intercept is redundant. Releveling changes coefficient labels, not fitted values for the same model space.

<a id="theory-05"></a>

### Interactions and separate equations
For $\beta_0+\beta_1x+\gamma D+\delta xD$, D=0 has intercept $\beta_0$ and slope $\beta_1$, while D=1 has intercept $\beta_0+\gamma$ and slope $\beta_1+\delta$. The group effect is $\gamma+\delta x$, so there is no single constant difference when slopes differ. Retain component terms when interpreting a hierarchical interaction model.

<a id="theory-06"></a>

### Fit, prediction and model selection
Adding predictors cannot increase OLS training SSE, so ordinary R² tends to rise. Adjusted $R^2=1-[SSE/(n-p)]/[SST/(n-1)]$ penalizes coefficient count, but is not an out-of-sample guarantee. Check residuals, predictor overlap, influential observations and validation appropriate to the data structure.

<a id="theory-07"></a>

### Reporting
Give the full equation, units, reference categories, interaction terms, sample size, uncertainty and diagnostics. A production-cost example should distinguish response units from predictor units and avoid interpreting a coefficient beyond observed combinations. The local source handouts labelled STA 519 inform these extensions; no handout images or copied text are included.

## Concepts covered

**Course points:** Multiple regression; partial slopes; dummy coding/reference category; interactions and prediction.

**Mission:** Predict what the formula will do, then run the simulation.

## Worked calculation · Dummy variables and interactions: design-matrix table

### Question and assumptions

A fitted illustrative model is $\hat y=10+2x+3D+xD$, where D=0 is the reference group and D=1 is the other group.

### Procedure

1. Create the intercept, x, D and xD columns.
2. Multiply each column by its stated coefficient.
3. Add contributions to obtain fitted values and compare group-specific intercepts and slopes.

### Working tables

These tables are calculated from the original teaching data below. They are visible before running Python and on GitHub.

| Intercept col | x | D | xD | 10×1 | 2x | 3D | 1×xD | Fitted y |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 1 | 1 | 0 | 0 | 10 | 2 | 0 | 0 | 12 |
| 1 | 2 | 0 | 0 | 10 | 4 | 0 | 0 | 14 |
| 1 | 1 | 1 | 1 | 10 | 2 | 3 | 1 | 16 |
| 1 | 2 | 1 | 2 | 10 | 4 | 3 | 2 | 19 |

| Group | Fitted equation | Intercept | Slope |
| --- | --- | --- | --- |
| Reference D=0 | 10 + 2x | 10 | 2 |
| Other D=1 | 13 + 3x | 13 | 3 |

In [ ]:

from math import comb, exp, factorial, log, sqrt
from statistics import mean, median, NormalDist
from itertools import product, combinations

def average_ranks(values):
    """Assign ranks 1..n, averaging every tie group."""
    return [1 + sum(v < x for v in values) + (sum(v == x for v in values)-1)/2
            for x in values]

def show_table(headers, rows):
    def fmt(x):
        if isinstance(x, float): return f"{x:.5g}"
        return str(x).replace("|", "/")
    lines = ["| " + " | ".join(map(str, headers)) + " |",
             "| " + " | ".join("---" for _ in headers) + " |"]
    lines += ["| " + " | ".join(fmt(x) for x in row) + " |" for row in rows]
    result = "\n".join(lines)
    try:
        from IPython.display import Markdown, display
        display(Markdown(result))
    except ImportError:
        print(result)


rows=[]
for x,D in [(1,0),(2,0),(1,1),(2,1)]:
    rows.append([1,x,D,x*D,10,2*x,3*D,x*D,10+2*x+3*D+x*D])
show_table(["Intercept col","x","D","xD","10×1","2x","3D","1×xD","Fitted y"],rows)
show_table(["Group","Fitted equation","Intercept","Slope"],[["Reference D=0","10 + 2x",10,2],["Other D=1","13 + 3x",13,3]])
assert [r[-1] for r in rows]==[12,14,16,19]


### Decision and interpretation

These coefficients are supplied to illustrate interpretation; this is not a coefficient-estimation example. Omitting xD forces parallel lines. With an intercept, using indicators for every category creates a redundant design column.

### Your calculation check

Reproduce the table on paper, change one input in the code, and explain which intermediate columns and conclusions change. Keep the test direction, zero/tie convention and assumptions explicit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
rng=np.random.default_rng(42)
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
n=140; temp=rng.normal(65,10,n); prod=rng.normal(150,25,n); days=rng.integers(18,24,n); D=rng.integers(0,2,n)
y=12+.18*temp+.31*prod+1.4*days+5*D+.015*temp*D+rng.normal(0,8,n)
X=np.c_[np.ones(n),temp,prod,days,D]; b=np.linalg.lstsq(X,y,rcond=None)[0]
Xi=np.c_[X,temp*D]; bi=np.linalg.lstsq(Xi,y,rcond=None)[0]
print('intercept,temp,prod,days,dummy:',np.round(b,2),'interaction slope:',round(bi[-1],3))
print('SSE parallel / interaction',np.sum((y-X@b)**2).round(1),np.sum((y-Xi@bi)**2).round(1))

### Practice set · Multiple regression
1. Interpret β₂=−4 for a night-shift indicator in a model controlling for production and temperature.
2. For groups A/B, write the mean lines if β₀=10, β₁=2, β₂=3, β₃=−.5 in Y=β₀+β₁x+β₂D+β₃xD.
3. How many dummy variables are needed for a 5-level factor with an intercept?
4. Explain multicollinearity’s effect on slope precision.

<details><summary>Answer sketches</summary>
1. At fixed modeled covariates, night shift is associated with 4 lower response units than the reference shift. 2. A: 10+2x; B: 13+1.5x. 3. Four. 4. Correlated predictors inflate variance/SEs and make partial effects hard to separate.
</details>

### Debrief
Interpret the dummy coefficient at fixed covariates. What does the interaction add?

Record the formula, assumptions, and one follow-up question.